In [ ]:
!pip install openai-whisper

In [2]:
import whisper
model = whisper.load_model("large-v3")

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import shutil

source_dir = '/content/drive/My Drive/'
dest_dir = '/content/'

print(f"Attempting to copy files from {source_dir} to {dest_dir}")

files_copied_count = 0
for filename in os.listdir(source_dir):
    if filename.startswith('clip_') and filename.endswith('.wav'):
        source_path = os.path.join(source_dir, filename)
        dest_path = os.path.join(dest_dir, filename)
        try:
            shutil.copy(source_path, dest_path)
            print(f'Copied: {filename}')
            files_copied_count += 1
        except Exception as e:
            print(f'Error copying {filename}: {e}')

if files_copied_count > 0:
    print(f'Successfully copied {files_copied_count} audio files.')
    print('Verifying copied files in /content:')
    !ls /content/clip_*.wav
else:
    print('No audio files starting with "clip_" and ending with ".wav" were found or copied.')

After running the cell above, please check the output to confirm that all your `clip_*.wav` files have been successfully copied. Once confirmed, you can proceed to run the transcription cell (`CrUV4s26LDQn`).

In [6]:
import json
import os
from tqdm.notebook import tqdm # Import tqdm for progress bar

results = []
audio_folder = "/content"  # wherever you uploaded them

# Filter and sort audio files numerically to process only a subset (e.g., the first 6)
all_audio_files = sorted([f for f in os.listdir(audio_folder) if f.startswith('clip_') and f.endswith(('.wav', '.mp3', '.m4a', '.flac'))],
                          key=lambda x: int(x.split('_')[1].split('.')[0]))
files_to_process = all_audio_files[:5] # Process the first 5 files

print(f"Processing {len(files_to_process)} audio files...")

for filename in tqdm(files_to_process, desc="Transcribing audio"):
    filepath = os.path.join(audio_folder, filename)
    try:
        result = model.transcribe(filepath)
        results.append({
            "audio_path": filename,
            "whisper_transcript": result["text"]
        })
        print(f"\n{filename}: {result['text']}") # Add newline for cleaner output with tqdm
    except Exception as e:
        print(f"\nError transcribing {filename}: {e}")

# Save so you can download it
with open("whisper_transcripts.json", "w") as f:
    json.dump(results, f, indent=2)

print("Transcription complete. Results saved to whisper_transcripts.json")

Processing 5 audio files...


Transcribing audio:   0%|          | 0/5 [00:00<?, ?it/s]


clip_0.wav:  ཱཱཡནཾཡཋཱཽ�ི཈ཱཱཀཱ་ཌྷཱསཁཾཱཇཱཱཽཇཱཁཽགཋིཁཱ཰ྋཱཌྷཱཉཾ཈ཱཁཱཆཱཱགཱཇཱཋཱ཈ཱཱདྷཱཁཱཆཱཁཱགཱཱཆཱཱཋ� � feminism. A, did anybody in your maybe working place, or your Having the age, yes, my coworker, he's an upright patient down gold shoe, Wilson hospital. No, I didn't come at. 4000 Inome are you married yes okay which are you with how many children two boys two boys okay i is uh you and your family living together no no no okay okay did you hit well yes okay okay did you smoke no no okay did you take um alcohol no no okay are you allergies to any drug or food no no okay all right you're welcome mr solomon my makeup blessing i will keep in touch with you thank you for your time okay

clip_1.wav:  you hear. I have high blood pressure in my right, discomfition. v I have a from possa ane Well, how long he started bleeding? target two days ago. । Okay,can you tell me more about this? The bleeding started two days ago I feel pain I have bright red blood. I noticed it on toilet paper when I wanted to po

In [7]:
def transcribe_whisper(audio_path: str) -> str:
    """
    Whisper ran externally in Google Colab (free GPU) since the dev machine
    has no GPU and an older CPU. Transcripts were generated in Colab and
    downloaded as whisper_transcripts.json — this function just looks up
    the matching pre-computed transcript by filename.
    """
    import json
    from pathlib import Path

    lookup_path = Path(__file__).resolve().parent.parent / "data" / "synthetic" / "whisper_transcripts.json"
    transcripts = json.loads(lookup_path.read_text())

    filename = Path(audio_path).name
    match = next((t for t in transcripts if t["audio_path"] == filename), None)

    if match is None:
        raise ValueError(f"No pre-computed Whisper transcript found for {filename}")

    return match["whisper_transcript"]

Once your Drive is mounted, you can copy your audio files from a specific folder in your Drive to the `/content` directory. Replace `Your_Audio_Files_Folder` with the actual name of the folder where your audio files are located in your Google Drive.

After copying, you can then rerun the transcription cell (`CrUV4s26LDQn`) to process the newly available audio files.